# 05 — Synthèse et discussion

Consolidation des deux axes et préparation des tableaux du mémoire.

## 1. Configuration

In [1]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW=PROJECT_ROOT/"data"/"raw"; DATA_PROCESSED=PROJECT_ROOT/"data"/"processed"
FIGURES=PROJECT_ROOT/"reports"/"figures"; RESULTS=PROJECT_ROOT/"reports"/"results"; MODELS=PROJECT_ROOT/"models"
for p in [DATA_PROCESSED,FIGURES,RESULTS,MODELS]: p.mkdir(parents=True,exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RANDOM_STATE=42
pd.set_option("display.max_columns",100)

## 2. Charger les résultats

In [2]:
result_files={
    'Potabilité — CV':RESULTS/'potability_cv_metrics.csv',
    'Potabilité — Test':RESULTS/'potability_test_metrics.csv',
    'Agressivité — CV':RESULTS/'aggressiveness_cv_metrics.csv',
    'Agressivité — Test':RESULTS/'aggressiveness_test_metrics.csv'}
loaded={}
for label,path in result_files.items():
    if path.exists():
        loaded[label]=pd.read_csv(path); print(label); display(loaded[label])
    else: print(label,': absent — exécuter les notebooks 03 et 04')

Potabilité — CV


,model,metric,mean,std
0,Logistic Regression,accuracy,0.494656,0.016121
1,Logistic Regression,precision,0.378256,0.019196
2,Logistic Regression,recall,0.461903,0.045570
3,Logistic Regression,f1,0.415525,0.028335
4,Logistic Regression,roc_auc,0.476758,0.018069
5,Random Forest,accuracy,0.674046,0.010341
6,Random Forest,precision,0.684389,0.022834
7,Random Forest,recall,0.306255,0.040162
8,Random Forest,f1,0.421454,0.038466
9,Random Forest,roc_auc,0.692662,0.010205


Potabilité — Test


,model,accuracy,precision,recall,f1,roc_auc
0,SVM-RBF,0.621951,0.515873,0.507812,0.511811,0.643975
1,Logistic Regression,0.524390,0.414634,0.531250,0.465753,0.547432
2,XGBoost,0.649390,0.584416,0.351562,0.439024,0.651797
3,Random Forest,0.672256,0.695238,0.285156,0.404432,0.658579


Agressivité — CV


,model,metric,mean,std
0,Logistic Regression,accuracy,0.826667,0.121228
1,Logistic Regression,precision,0.932481,0.114544
2,Logistic Regression,recall,0.725000,0.171998
3,Logistic Regression,f1,0.808259,0.140373
4,Logistic Regression,roc_auc,0.949603,0.077595
5,Random Forest,accuracy,0.906667,0.045325
6,Random Forest,precision,0.918256,0.047908
7,Random Forest,recall,0.908333,0.066667
8,Random Forest,f1,0.911489,0.044056
9,Random Forest,roc_auc,0.964087,0.030054


Agressivité — Test


,model,accuracy,precision,recall,f1,roc_auc
0,XGBoost,0.929825,0.906250,0.966667,0.935484,0.962963
1,Random Forest,0.894737,0.900000,0.900000,0.900000,0.945062
2,Logistic Regression,0.859649,0.958333,0.766667,0.851852,0.965432
3,SVM-RBF,0.789474,0.950000,0.633333,0.760000,0.946914


## 3. Meilleurs modèles

In [3]:
rows=[]
for axe,key in [('Potabilité','Potabilité — Test'),('Agressivité chimique','Agressivité — Test')]:
    if key in loaded and not loaded[key].empty:
        r=loaded[key].sort_values(['f1','roc_auc'],ascending=False).iloc[0].to_dict(); r['axe']=axe; rows.append(r)
summary=pd.DataFrame(rows)
if not summary.empty:
    cols=[c for c in ['axe','model','accuracy','precision','recall','f1','roc_auc'] if c in summary.columns]
    display(summary[cols].round(4)); summary[cols].to_csv(RESULTS/'synthese_modeles.csv',index=False)

,axe,model,accuracy,precision,recall,f1,roc_auc
0,Potabilité,SVM-RBF,0.6220,0.5159,0.5078,0.5118,0.644
1,Agressivité chimique,XGBoost,0.9298,0.9062,0.9667,0.9355,0.963


## 4. Lecture scientifique

**Potabilité :** classification directe de `Potability`, imputation dans le pipeline et comparaison de modèles.

**Agressivité :** classe construite à partir de Larson. Le lien direct entre la cible et Cl/SO4/HCO3 doit être reconnu.

**Langelier :** non calculé sans température mesurée.

## 5. Mise en perspective dessalement

**Eau brute → prétraitement → osmose inverse → reminéralisation → contrôle de qualité → risque d’agressivité/corrosion → aide à la décision par science des données.**

## 6. Limites

1. Les fichiers fournis ne sont pas identifiés comme des mesures directes d’Al Hoceima.
2. Le CSV de potabilité ne contient pas de variables de procédé d’osmose inverse.
3. `Y` dans le classeur n’est pas documentée.
4. Température absente pour le LSI.
5. Cible Larson dérivée de certaines variables d’entrée.
6. Pas de conclusion possible sur l’état des membranes avec ces deux datasets seuls.

## 7. Tableau final

In [4]:
conclusion=pd.DataFrame({'Axe':['Potabilité','Agressivité chimique'],'Cible':['Potability (0/1)','Larson_corrosive (IC >= 1)'],'Meilleur modèle':[None,None],'F1 test':[None,None],'ROC-AUC test':[None,None]})
if 'summary' in globals() and not summary.empty:
    for _,r in summary.iterrows():
        mask=conclusion['Axe'].eq(r['axe']); conclusion.loc[mask,'Meilleur modèle']=r.get('model'); conclusion.loc[mask,'F1 test']=r.get('f1'); conclusion.loc[mask,'ROC-AUC test']=r.get('roc_auc')
display(conclusion); conclusion.to_csv(RESULTS/'tableau_conclusion.csv',index=False)

,Axe,Cible,Meilleur modèle,F1 test,ROC-AUC test
0,Potabilité,Potability (0/1),SVM-RBF,0.511811,0.643975
1,Agressivité chimique,Larson_corrosive (IC >= 1),XGBoost,0.935484,0.962963


## 8. Conclusion générale

Répondre à la problématique sans dépasser ce que démontrent les données : classification de potabilité, automatisation de l’évaluation Larson, et conditions nécessaires pour une transposition industrielle complète à Al Hoceima.